<a href="https://colab.research.google.com/github/UMA-GitHub-alt/lab_assignments/blob/main/%E7%A0%94%E7%A9%B6%E5%AE%A4%E8%AA%B2%E9%A1%8C3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import glob
import numpy as np
import re

def load_data(filepath):
    """
    タブ区切りの時系列データを読み込む関数。
    Level 1(1次元)からLevel 4(多次元)まで一貫して処理できるよう、
    1次元データも(N, 1)の2次元配列に変換します。
    """
    data = np.loadtxt(filepath)
    if data.ndim == 1:
        data = data.reshape(-1, 1)
    return data

def dtw_distance(ts_a, ts_b):
    """
    動的時間伸縮法(DTW)による2つの時系列データ間の距離を計算する関数(スクラッチ実装)。
    多次元・可変長に対応しています。
    ts_a: shape (N, 次元数)
    ts_b: shape (M, 次元数)
    """
    N, M = len(ts_a), len(ts_b)

    # 累積距離行列を無限大で初期化
    cost_matrix = np.full((N + 1, M + 1), np.inf)
    cost_matrix[0, 0] = 0

    # 動的計画法（DP）による最小累積距離の計算
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            # 各時点でのユークリッド距離（多次元のベクトル間距離）
            dist = np.linalg.norm(ts_a[i-1] - ts_b[j-1])

            # 直前の3方向（挿入、欠失、一致）から最小コストを選択して加算
            cost_matrix[i, j] = dist + min(
                cost_matrix[i-1, j],   # ts_bの時間を進める (上)
                cost_matrix[i, j-1],   # ts_aの時間を進める (左)
                cost_matrix[i-1, j-1]  # 両方の時間を進める (斜め)
            )

    # 最終的なDTW距離を返す
    return cost_matrix[N, M]

def main():
    # フォルダパスの設定 (環境に合わせて変更してください)
    ref_dir = '/content/drive/MyDrive/labassignment3/level4/reference'
    test_dir = '/content/drive/MyDrive/labassignment3/level4/test'

    print("--- リファレンスデータの読み込み ---")
    # クラス1とクラス2の基準データを読み込む
    ref_class1_list = []
    ref_class2_list = []
    # --- クラス1（フォルダ '1'）のデータをすべて読み込む ---
    # os.path.join(ref_dir, '1', '*.dat') で フォルダ1の中の全datファイルを取得
    for file_path in glob.glob(os.path.join(ref_dir, '1', '*.dat')):
      ref_class1_list.append(load_data(file_path))

    # --- クラス2（フォルダ '2'）のデータをすべて読み込む ---
    for file_path in glob.glob(os.path.join(ref_dir, '2', '*.dat')):
      ref_class2_list.append(load_data(file_path))

    print(f"クラス1の基準データ数: {len(ref_class1_list)}")
    print(f"クラス2の基準データ数: {len(ref_class2_list)}")
    print("-" * 30)

    # テストデータのファイル一覧を取得し、ファイル名順にソート
    test_files = glob.glob(os.path.join(test_dir, '*.dat'))
    test_files = sorted(test_files, key=lambda x: int(re.search(r'\d+', os.path.basename(x)).group()))

    if not test_files:
        print("テストデータが見つかりません。フォルダ構成を確認してください。")
        return

    print("--- テストデータの分類開始 ---")
    correct_count = 0  # 答え合わせ用（正解がわかっている場合）

    for test_file in test_files:
        test_data = load_data(test_file)
        filename = os.path.basename(test_file)

        # DTWで基準データとの距離をそれぞれ計算
        dist_to_class1 = dtw_distance(test_data, ref_class1_list)
        dist_to_class2 = dtw_distance(test_data, ref_class2_list)

        # 距離が近い方（コストが小さい方）を予測クラスとする (1-NN)
        predicted_class = 1 if dist_to_class1 < dist_to_class2 else 2

        print(f"File: {filename} -> 予測クラス: {predicted_class} "
              f"(Dist1: {dist_to_class1:.2f}, Dist2: {dist_to_class2:.2f})")

if __name__ == "__main__":
    main()

--- リファレンスデータの読み込み ---
クラス1の基準データ数: 3
クラス2の基準データ数: 3
------------------------------
--- テストデータの分類開始 ---
File: data1.dat -> 予測クラス: 1 (Dist1: 3180.10, Dist2: 3347.70)
File: data2.dat -> 予測クラス: 1 (Dist1: 3222.21, Dist2: 3659.80)
File: data3.dat -> 予測クラス: 1 (Dist1: 2979.75, Dist2: 3173.18)
File: data4.dat -> 予測クラス: 1 (Dist1: 2980.45, Dist2: 4154.92)
File: data5.dat -> 予測クラス: 1 (Dist1: 2976.42, Dist2: 4024.71)
File: data6.dat -> 予測クラス: 1 (Dist1: 3205.49, Dist2: 4242.97)
File: data7.dat -> 予測クラス: 1 (Dist1: 2791.24, Dist2: 3385.58)
File: data8.dat -> 予測クラス: 2 (Dist1: 3496.22, Dist2: 2983.32)
File: data9.dat -> 予測クラス: 2 (Dist1: 4341.67, Dist2: 4036.74)
File: data10.dat -> 予測クラス: 2 (Dist1: 3022.51, Dist2: 3015.60)
File: data11.dat -> 予測クラス: 2 (Dist1: 3607.12, Dist2: 3046.90)
File: data12.dat -> 予測クラス: 2 (Dist1: 3832.49, Dist2: 3441.22)
File: data13.dat -> 予測クラス: 1 (Dist1: 4362.18, Dist2: 4366.06)
File: data14.dat -> 予測クラス: 2 (Dist1: 4483.25, Dist2: 3603.32)


In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import re
from torch.nn.utils.rnn import pad_sequence

# --- 1. モデルの定義 (LSTM) ---
class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(TimeSeriesLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTMレイヤー: batch_first=Trueで入力を(Batch, Seq, Feature)の形にする
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # 全結合レイヤー: LSTMの最後の出力をクラス数に変換
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # LSTMにデータを入力
        out, _ = self.lstm(x)
        # 最後のタイムステップの出力だけを取り出して全結合層へ
        out = self.fc(out[:, -1, :])
        return out

# --- 2. データの読み込み補助 ---
def load_data(filepath):
    data = np.loadtxt(filepath)
    if data.ndim == 1:
        data = data.reshape(-1, 1) # 1次元データを(N, 1)に
    return torch.tensor(data, dtype=torch.float32)

def main():
    # --- 設定 ---
    ref_dir = '/content/drive/MyDrive/labassignment3/level4/reference'
    test_dir = '/content/drive/MyDrive/labassignment3/level4/test'

    # データの次元数（Level 4は64次元）
    input_dim = 64
    hidden_dim = 32
    num_epochs = 100

    print("--- データの準備と学習（Training） ---")

    ref_class1_list = []
    ref_class2_list = []

    # --- クラス1（フォルダ '1'）のデータをすべて読み込む ---
    for file_path in glob.glob(os.path.join(ref_dir, '1', '*.dat')):
        ref_class1_list.append(load_data(file_path))

    # --- クラス2（フォルダ '2'）のデータをすべて読み込む ---
    for file_path in glob.glob(os.path.join(ref_dir, '2', '*.dat')):
        ref_class2_list.append(load_data(file_path))

    # データ読み込みエラーのチェック
    if len(ref_class1_list) == 0 and len(ref_class2_list) == 0:
        raise ValueError(f"データが読み込めませんでした。パス ({ref_dir}) を確認してください。")

    # --- PyTorchの学習用にデータをまとめる ---
    # 1. データを1つのリストに結合
    X_train = ref_class1_list + ref_class2_list

    # 2. 正解ラベルの作成 (クラス1の数だけ[0]を、クラス2の数だけ[1]を作る)
    y_train = [0] * len(ref_class1_list) + [1] * len(ref_class2_list)

    # 3. リストをPyTorchのテンソルに変換
    X_train_tensor = torch.stack(X_train)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)

    # モデル、損失関数、最適化手法の初期化
    model = TimeSeriesLSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=1, num_classes=2)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # 学習ループ
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()

        # 予測と誤差の計算
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)

        # 重みの更新
        loss.backward()
        optimizer.step()

        if (epoch+1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    print("--- テストデータの予測（Inference） ---")
    model.eval() # 推論モードに切り替え
    test_files = glob.glob(os.path.join(test_dir, '*.dat'))
    test_files = sorted(test_files, key=lambda x: int(re.search(r'\d+', os.path.basename(x)).group()))

    with torch.no_grad(): # 推論時は勾配計算をしない
        for test_file in test_files:
            x_test = load_data(test_file).unsqueeze(0) # バッチ次元を追加
            filename = os.path.basename(test_file)

            output = model(x_test)
            _, predicted = torch.max(output.data, 1)

            # ラベル0ならクラス1、ラベル1ならクラス2
            pred_class = 1 if predicted.item() == 0 else 2
            print(f"File: {filename} -> 予測クラス: {pred_class}")

if __name__ == "__main__":
    main()

--- データの準備と学習（Training） ---
Epoch [20/100], Loss: 0.0270
Epoch [40/100], Loss: 0.0056
Epoch [60/100], Loss: 0.0015
Epoch [80/100], Loss: 0.0008
Epoch [100/100], Loss: 0.0006
--- テストデータの予測（Inference） ---
File: data1.dat -> 予測クラス: 1
File: data2.dat -> 予測クラス: 2
File: data3.dat -> 予測クラス: 1
File: data4.dat -> 予測クラス: 1
File: data5.dat -> 予測クラス: 1
File: data6.dat -> 予測クラス: 1
File: data7.dat -> 予測クラス: 1
File: data8.dat -> 予測クラス: 2
File: data9.dat -> 予測クラス: 1
File: data10.dat -> 予測クラス: 2
File: data11.dat -> 予測クラス: 2
File: data12.dat -> 予測クラス: 2
File: data13.dat -> 予測クラス: 1
File: data14.dat -> 予測クラス: 2
